# TradePulse — Transformer Training on Colab T4 GPU

**Steps:**
1. Upload `train.csv` and `test.csv` from your `data/` folder
2. Run all cells in order
3. Download the trained model and predictions at the end

**Runtime:** ~15-20 minutes on T4 GPU

In [ ]:
# Cell 1 — Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers==4.40.0 datasets accelerate scikit-learn joblib

In [ ]:
# Cell 3 — Upload train.csv and test.csv
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
os.makedirs('models/transformer_bilateral', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('nlp', exist_ok=True)

print('Upload train.csv and test.csv from your data/ folder')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'data/{fname}', 'wb') as f:
        f.write(content)
    print(f'Saved data/{fname}')

In [ ]:
# Cell 4 — Create label_mapping.py
os.makedirs('nlp', exist_ok=True)
with open('nlp/__init__.py', 'w') as f:
    f.write('')

with open('nlp/label_mapping.py', 'w') as f:
    f.write('''"""Fixed label order for sklearn + Hugging Face."""
from __future__ import annotations
LABELS: list[str] = ["adversarial", "cooperative", "neutral"]
LABEL2ID: dict[str, int] = {lab: i for i, lab in enumerate(LABELS)}
ID2LABEL: dict[int, str] = {i: lab for i, lab in enumerate(LABELS)}
''')
print('Created nlp/label_mapping.py')

In [ ]:
# Cell 5 — Training script
TRAIN_SCRIPT = '''
from __future__ import annotations
import csv, sys, argparse
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)

sys.path.insert(0, ".")
from nlp.label_mapping import ID2LABEL, LABEL2ID, LABELS

DATA   = Path("data")
MODELS = Path("models/transformer_bilateral")
RESULTS = Path("results")
MODEL_NAME = "distilbert-base-uncased"

def _text(row):
    c1 = (row.get("country_1") or "").strip().upper()
    c2 = (row.get("country_2") or "").strip().upper()
    text = (row.get("text") or row.get("headline") or "").strip()
    if c1 and c2:
        return f"{c1}-{c2}: {text}"
    return text

def _load(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

train_rows = _load(DATA / "train.csv")
test_rows  = _load(DATA / "test.csv")
print(f"Train: {len(train_rows)} | Test: {len(test_rows)}")

labels_list = [r["label"].strip().lower() for r in train_rows]
idx = list(range(len(train_rows)))
tr_idx, va_idx = train_test_split(idx, test_size=0.15, random_state=42, stratify=labels_list)
fit_rows = [train_rows[i] for i in tr_idx]
val_rows = [train_rows[i] for i in va_idx]

def make_ds(rows):
    return Dataset.from_dict({
        "text":   [_text(r) for r in rows],
        "labels": [LABEL2ID[r["label"].strip().lower()] for r in rows],
    })

ds_train = make_ds(fit_rows)
ds_val   = make_ds(val_rows)
ds_test  = make_ds(test_rows)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

ds_train = ds_train.map(tokenize, batched=True, remove_columns=["text"])
ds_val   = ds_val.map(tokenize,   batched=True, remove_columns=["text"])
ds_test  = ds_test.map(tokenize,  batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }

MODELS.mkdir(parents=True, exist_ok=True)
targs = TrainingArguments(
    output_dir=str(MODELS),
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to=[],
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=targs,
    train_dataset=ds_train, eval_dataset=ds_val,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model(str(MODELS))
tokenizer.save_pretrained(str(MODELS))

pred_out = trainer.predict(ds_test)
logits = pred_out.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
pred_ids = np.argmax(logits, axis=-1)

RESULTS.mkdir(parents=True, exist_ok=True)
fields = ["id","true_label","pred_label","p_adversarial","p_cooperative","p_neutral"]
with open(RESULTS / "transformer_test_predictions.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for i, row in enumerate(test_rows):
        w.writerow({
            "id": row.get("id",""),
            "true_label": row["label"],
            "pred_label": LABELS[int(pred_ids[i])],
            "p_adversarial": f"{probs[i,0]:.6f}",
            "p_cooperative": f"{probs[i,1]:.6f}",
            "p_neutral":     f"{probs[i,2]:.6f}",
        })

from sklearn.metrics import classification_report
gold = [r["label"].strip().lower() for r in test_rows]
pred = [LABELS[int(p)] for p in pred_ids]
print("\\n=== TEST SET RESULTS ===")
print(classification_report(gold, pred, digits=4))
print(f"Macro F1: {f1_score(gold, pred, average=\'macro\'):.4f}")
print(f"Saved model -> {MODELS}")
print(f"Saved preds -> {RESULTS}/transformer_test_predictions.csv")
'''

with open('train_transformer_colab.py', 'w') as f:
    f.write(TRAIN_SCRIPT)
print('Training script written.')

In [ ]:
# Cell 6 — RUN TRAINING (15-20 min on T4)
%run train_transformer_colab.py

In [ ]:
# Cell 7 — Plot training curves
import json, glob
import matplotlib.pyplot as plt

# Find trainer_state.json
state_files = glob.glob('models/transformer_bilateral/**/trainer_state.json', recursive=True)
state = json.loads(open(state_files[0]).read())
log = state['log_history']

train_steps, train_loss = [], []
eval_steps, eval_loss, eval_acc, eval_f1 = [], [], [], []

for e in log:
    if 'loss' in e and 'eval_loss' not in e:
        train_steps.append(e['step']); train_loss.append(e['loss'])
    if 'eval_loss' in e:
        eval_steps.append(e['step']); eval_loss.append(e['eval_loss'])
        eval_acc.append(e.get('eval_accuracy', 0))
        eval_f1.append(e.get('eval_macro_f1', 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TradePulse — DistilBERT Training (5 epochs)', fontsize=13, fontweight='bold')

axes[0].plot(train_steps, train_loss, label='Train Loss', color='#2196F3', alpha=0.7)
axes[0].plot(eval_steps, eval_loss, label='Val Loss', color='#F44336', linewidth=2.5, marker='o')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')

epochs = list(range(1, len(eval_acc)+1))
axes[1].plot(epochs, [a*100 for a in eval_acc], label='Val Accuracy %', color='#4CAF50', linewidth=2.5, marker='o')
axes[1].plot(epochs, [f*100 for f in eval_f1], label='Val Macro F1 %', color='#FF9800', linewidth=2.5, marker='s')
axes[1].set_title('Validation Metrics per Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score (%)'); axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val Macro F1: {max(eval_f1)*100:.2f}%')
print(f'Best val Accuracy: {max(eval_acc)*100:.2f}%')

In [ ]:
# Cell 8 — Confusion matrix
import csv
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score

labels_order = ['adversarial', 'cooperative', 'neutral']
gold, pred = [], []
with open('results/transformer_test_predictions.csv', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        gold.append(row['true_label'].strip().lower())
        pred.append(row['pred_label'].strip().lower())

cm = confusion_matrix(gold, pred, labels=labels_order)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='% of true class')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels([l.capitalize() for l in labels_order])
ax.set_yticklabels([l.capitalize() for l in labels_order])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Transformer (Test Set)')
for i in range(3):
    for j in range(3):
        color = 'white' if cm_pct[i,j] > 60 else 'black'
        ax.text(j, i, f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)',
                ha='center', va='center', fontsize=11, color=color, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Classification Report ===')
print(classification_report(gold, pred, labels=labels_order, digits=4))
print(f'Macro F1: {f1_score(gold, pred, average="macro"):.4f}')

In [ ]:
# Cell 9 — Download everything
import shutil
from google.colab import files

# Zip the model
shutil.make_archive('transformer_bilateral', 'zip', 'models/transformer_bilateral')
files.download('transformer_bilateral.zip')

# Download predictions
files.download('results/transformer_test_predictions.csv')

# Download plots
files.download('training_curves.png')
files.download('confusion_matrix.png')

print('Downloaded: model zip, predictions, training curves, confusion matrix')
print('Unzip transformer_bilateral.zip into your local models/transformer_bilateral/ folder')